In [26]:
import requests
import json
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta

# Load environment variables from .env file
load_dotenv('secrets.env')

True

In [27]:
# Google Authentication
PROJECT_ID = os.getenv('PROJECT_ID')
DATASET_ID = os.getenv('DATASET_ID')
TABLE_ID = 'vend_sales'  # Name the table vend_sales
TEMP_TABLE_ID = 'vend_sales_temp'  # Temporary table for deduplication

# Get the path to your service account key file from environment variable
SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')

# Initialize BigQuery client
credentials = service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE)
client = bigquery.Client(credentials=credentials, project=PROJECT_ID)

In [28]:
# Define the BigQuery table references
dataset_ref = client.dataset(DATASET_ID)
table_ref = dataset_ref.table(TABLE_ID)
temp_table_ref = dataset_ref.table(TEMP_TABLE_ID)

In [36]:
# Function to fetch data from Vend API
def fetch_vend_data(url, headers, params=None):
    response = requests.get(url, headers=headers, params=params)
    response.raise_for_status()
    return response.json()

# Function to load data into BigQuery
def load_data_to_bigquery(client, table_ref, rows_to_insert):
    errors = client.insert_rows_json(table_ref, rows_to_insert)
    if errors:
        print(f"Encountered errors while inserting rows: {errors}")
    else:
        print("Data successfully inserted into BigQuery.")

# Function to get the max created date from BigQuery
def get_max_created_date(client, dataset_id, table_id):
    query = f"""
    SELECT DATE(sale_date) as sale_date
    FROM `{PROJECT_ID}.{dataset_id}.{table_id}`
    ORDER BY created_at DESC
    LIMIT 1
    """
    query_job = client.query(query)
    results = query_job.result()
    for row in results:
        return row.sale_date

In [37]:
# Function to check if records already exist in BigQuery
def check_existing_records(client, dataset_id, table_id, sale_date):
    query = f"""
    SELECT id
    FROM `{PROJECT_ID}.{dataset_id}.{table_id}`
    WHERE sale_date = '{sale_date}'
    """
    query_job = client.query(query)
    existing_ids = {row.id for row in query_job.result()}
    return existing_ids

In [38]:
# Vend API details
vend_url = "https://ashcorp.retail.lightspeed.app/api/2.0/search"
vend_headers = {
    "accept": "application/json",
    "authorization": f"Bearer {os.getenv('LIGHTSPEED_ACCESS_TOKEN')}"
}


In [39]:
# Get the max created date from BigQuery
max_created_date = get_max_created_date(client, DATASET_ID, TABLE_ID)
if max_created_date is None:
    # If no data in BigQuery, set a default start date
    max_created_date = datetime(2020, 1, 1).date()

# Format the date for the API request
date_from = max_created_date.isoformat()


In [40]:
# Pagination parameters
offset = 0
page_size = 1000

In [41]:
# Extract, Transform, Load (ETL) process
while True:
    # Extract data from Vend API
    params = {
        "type": "sales",
        "page_size": page_size,
        "date_from": date_from,
        "offset": offset
    }
    data = fetch_vend_data(vend_url, vend_headers, params)
    
    # Transform data (if needed)
    rows_to_insert = data["data"]
    
    # Check for existing records in BigQuery
    existing_ids = check_existing_records(client, DATASET_ID, TABLE_ID, date_from)
    new_rows = [row for row in rows_to_insert if row["id"] not in existing_ids]
    
    # Load new data into BigQuery
    if new_rows:
        load_data_to_bigquery(client, table_ref, new_rows)
    
    # Check if there are more pages to fetch
    if len(rows_to_insert) < page_size:
        break
    
    # Increment the offset for the next request
    offset += page_size

print("Incremental ETL process with deduplication completed successfully.")

Data successfully inserted into BigQuery.
Incremental ETL process with deduplication completed successfully.
